# 04 — Aprendizaje no supervisado

Este notebook valida el flujo de K-Means sobre el dataset final del ETL.

El flujo reproducible oficial se ejecuta con `python -m src.models.run_clustering`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.cluster import (
    evaluate_kmeans,
    get_cluster_info,
    get_cluster_movies,
    get_recommendations,
    select_best_k,
    train_kmeans,
)

## 1. Cargar dataset final

In [ ]:
df = pd.read_csv('../data/final/movies_dataset.csv')
print(f'Dataset: {df.shape[0]} peliculas, {df.shape[1]} columnas')
df.head()

## 2. Evaluar k

In [ ]:
evaluation = evaluate_kmeans(df, k_values=range(2, 11), silhouette_sample_size=10000)
evaluation

## 3. Entrenar K-Means con el mejor k

In [ ]:
best_k = select_best_k(evaluation)
model_km, scaler, df_clustered = train_kmeans(df, n_clusters=best_k)
print(f'Clusters formados: {model_km.n_clusters}')
print(f'Inercia (inertia): {model_km.inertia_:.2f}')

In [ ]:
# Resumen de cada cluster
cluster_summary = get_cluster_info(df_clustered)
cluster_summary

In [ ]:
# Distribucion de peliculas por cluster
df_clustered['cluster'].value_counts().sort_index().plot(
    kind='bar', figsize=(10, 5), color=['#22d3ee', '#8b5cf6', '#f472b6', '#34d399', '#facc15']
)
plt.title('Distribucion de peliculas por cluster')
plt.xlabel('Cluster')
plt.ylabel('Cantidad de peliculas')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 peliculas del cluster 0 (mas votadas)
get_cluster_movies(df_clustered, cluster_label=0, top_n=5)

## 4. Recomendaciones por cluster

In [ ]:
source_movie = df_clustered.sort_values('numVotes', ascending=False).iloc[0]['tconst']
get_recommendations(df_clustered, source_movie, top_n=10)[[
    'primaryTitle', 'startYear', 'genres', 'averageRating', 'numVotes', 'cluster', 'distanceToCentroid'
]]

## 5. Generar artefactos reproducibles

In [ ]:
# Ejecutar en terminal para generar modelo, dataset clusterizado, tablas, figuras y reporte:
# python -m src.models.run_clustering